# Étape 3 — Analyse exploratoire des données

Dans ce notebook, nous explorons les données House Prices. Le contexte, la métrique et le dictionnaire des variables sont présentés à [l'étape 2 — Présentation du problème](02_presentation_probleme.ipynb).

L'analyse porte surtout sur `train.csv`, seul fichier contenant `SalePrice`. Nous examinons aussi `test.csv` pour comparer les valeurs manquantes et les catégories, sans jamais utiliser ce fichier pour définir une règle de préparation. Enfin, une corrélation ou une différence de prix entre deux groupes de maisons ne démontre pas une causalité : nos constats décrivent des associations dans les ventes d'Ames entre 2006 et 2010.

## Préparation

Cette cellule importe les bibliothèques et lit les fichiers. À l'étape 2, nous avions lu les CSV sous forme de texte pour vérifier les codes bruts. Ici, nous avons besoin de valeurs numériques : nous lisons donc les fichiers avec `na_values=["NA"]` et `keep_default_na=False`. Seul le code `NA` devient une valeur absente, tandis que `None` reste une catégorie, par exemple « aucun parement de maçonnerie » dans `MasVnrType`.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from matplotlib import MatplotlibDeprecationWarning
from matplotlib.patches import Patch
from matplotlib.ticker import NullFormatter

# Repérer la racine depuis le dossier courant ou l'un de ses parents.
current = Path.cwd().resolve()
root = next(
    (path for path in (current, *current.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dépôt MAP6014.")

data_dir = root / "data"
# Seul le code « NA » devient une valeur absente : « None » reste une catégorie.
read_options = {"na_values": ["NA"], "keep_default_na": False}
train = pd.read_csv(data_dir / "train.csv", **read_options)
test = pd.read_csv(data_dir / "test.csv", **read_options)
description_text = (data_dir / "data_description.txt").read_text(encoding="utf-8")

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert train["SalePrice"].notna().all() and train["SalePrice"].gt(0).all()
assert train["MasVnrType"].eq("None").any()

# Avertissement interne à seaborn, sans effet sur les figures.
warnings.filterwarnings("ignore", message="vert: bool", category=MatplotlibDeprecationWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

price = train["SalePrice"]
log_price = np.log(price)


def format_dollars(value):
    """Afficher un montant en dollars avec une espace comme séparateur de milliers."""
    return f"{value:,.0f} $".replace(",", " ")

def show_title(title):
    """Afficher dans les sorties le titre de la sous-section correspondante du texte."""
    display(Markdown(f"**{title}**"))


## 1. Description des données

Les données proviennent de la compétition Kaggle *House Prices – Advanced Regression Techniques* et décrivent des ventes résidentielles à Ames (Iowa) entre 2006 et 2010. `train.csv` contient un identifiant `Id`, 79 caractéristiques et le prix de vente `SalePrice`. `test.csv` contient les mêmes colonnes, sans le prix.

Le type informatique d'une colonne ne suffit pas à décrire sa signification. Nous regroupons donc les 79 caractéristiques selon leur nature, en nous appuyant sur `data_description.txt`. Nous comparons ensuite les codes présents dans les CSV avec ceux décrits dans le dictionnaire.

In [ ]:
# Regrouper les 79 caractéristiques selon leur nature (data_description.txt).
categorical_text = [
    column for column in train.columns
    if not pd.api.types.is_numeric_dtype(train[column])
]
variable_groups = {
    "Surfaces, longueurs et montants": [
        "LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
        "BsmtUnfSF", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF",
        "GrLivArea", "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch",
        "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal",
    ],
    "Comptes (pièces, salles de bain, voitures)": [
        "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
        "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars",
    ],
    "Années et mois": ["YearBuilt", "YearRemodAdd", "GarageYrBlt", "MoSold", "YrSold"],
    "Scores numériques de 1 à 10": ["OverallQual", "OverallCond"],
    "Code numérique nominal": ["MSSubClass"],
    "Catégories textuelles": categorical_text,
}
categorical_groups = {"Code numérique nominal", "Catégories textuelles"}
features = [column for group in variable_groups.values() for column in group]
assert len(features) == 79 and set(features) == set(train.columns) - {"Id", "SalePrice"}

numeric_features = [
    column
    for group, columns in variable_groups.items() if group not in categorical_groups
    for column in columns
]
categorical_features = categorical_text + ["MSSubClass"]

group_table = pd.DataFrame(
    [(group, "Catégorielle" if group in categorical_groups else "Numérique", len(columns),
      ", ".join(columns[:5]) + (", …" if len(columns) > 5 else ""))
     for group, columns in variable_groups.items()],
    columns=["Groupe", "Traitement", "Nombre", "Exemples"],
)
show_title("Groupes de caractéristiques selon leur nature")
display(group_table)

# Extraire les codes documentés de chaque variable textuelle dans data_description.txt.
documented_codes, current_variable = {}, None
for line in description_text.splitlines():
    header = re.match(r"^(\w+):", line)
    if header:
        current_variable = header.group(1)
        documented_codes[current_variable] = set()
        continue
    code = re.match(r"^\s+(\S[^\t]*?)\s*\t", line)
    if current_variable and code:
        documented_codes[current_variable].add(code.group(1).strip())

code_gaps = []
for column in categorical_text:
    observed = set(train[column].dropna()) | set(test[column].dropna())
    undocumented = observed - documented_codes.get(column, set())
    if undocumented:
        code_gaps.append({
            "Variable": column,
            "Codes présents dans les CSV": ", ".join(sorted(undocumented)),
            "Codes du dictionnaire non observés": ", ".join(
                sorted(documented_codes[column] - observed)
            ),
        })
show_title("Codes catégoriels absents du dictionnaire")
display(pd.DataFrame(code_gaps))

# Figure : nombre de caractéristiques par groupe, selon le traitement prévu.
colors = {"Numérique": "tab:blue", "Catégorielle": "tab:orange"}
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(group_table["Groupe"], group_table["Nombre"],
        color=group_table["Traitement"].map(colors))
for position, count in enumerate(group_table["Nombre"]):
    ax.text(count + 0.4, position, str(count), va="center")
ax.invert_yaxis()
ax.set_xlim(0, group_table["Nombre"].max() + 4)
ax.set_title("Répartition des 79 caractéristiques selon leur nature")
ax.set_xlabel("Nombre de variables")
ax.legend(handles=[
    Patch(color=color, label=f"{label} ({group_table.loc[group_table['Traitement'].eq(label), 'Nombre'].sum()})")
    for label, color in colors.items()
], title="Traitement prévu", loc="center right")
plt.tight_layout()
plt.show()

Les 79 caractéristiques se répartissent en **35 numériques** (surfaces, comptes, années, scores) et **44 catégorielles**. `MSSubClass`, bien que codée par des nombres, désigne un type de logement : nous la traiterons comme une catégorie. Quatre variables ont des codes écrits différemment du dictionnaire (ex. `Wd Shng` au lieu de `WdShing` dans `Exterior2nd`). Il faudra harmoniser `Exterior1st` et `Exterior2nd` avant de les combiner.

## 2. Statistiques descriptives

Nous résumons d'abord la variable cible `SalePrice`, puis huit caractéristiques numériques importantes. Pour les 44 variables catégorielles, nous mesurons le nombre de modalités et la part de la modalité la plus fréquente : une variable presque toujours égale à la même modalité apporte peu d'information. Nous repérons enfin les modalités présentes dans `test.csv` mais jamais observées dans `train.csv`.

In [ ]:
# Variable cible.
show_title("Statistiques descriptives du prix de vente")
display(pd.DataFrame({
    "Statistique": [
        "Nombre de maisons", "Moyenne", "Écart type", "Minimum", "1er quartile",
        "Médiane", "3e quartile", "Maximum",
        "Coefficient d'asymétrie", "Coefficient d'aplatissement (excès)",
    ],
    "SalePrice": [
        f"{price.count():,}".replace(",", " "),
        format_dollars(price.mean()), format_dollars(price.std()),
        format_dollars(price.min()), format_dollars(price.quantile(0.25)),
        format_dollars(price.median()), format_dollars(price.quantile(0.75)),
        format_dollars(price.max()), f"{price.skew():.2f}", f"{price.kurt():.2f}",
    ],
}).set_index("Statistique"))

# Boîte à moustaches de SalePrice.
box_style = dict(
    widths=0.5, patch_artist=True,
    boxprops={"facecolor": "tab:blue", "alpha": 0.35},
    medianprops={"color": "tab:blue", "linewidth": 2},
    flierprops={"marker": "o", "markersize": 3, "alpha": 0.4},
)
fig, ax = plt.subplots(figsize=(9, 2.2))
ax.boxplot(price, orientation="horizontal", **box_style)
ax.set_yticks([])
ax.xaxis.set_major_formatter(lambda value, _: format_dollars(value))
ax.set_title("Boîte à moustaches du prix de vente")
ax.set_xlabel("SalePrice")
plt.tight_layout()
plt.show()

# Principales caractéristiques numériques.
key_numeric = [
    "LotArea", "GrLivArea", "TotalBsmtSF", "GarageArea",
    "TotRmsAbvGrd", "GarageCars", "OverallQual", "YearBuilt",
]
show_title("Statistiques des principales caractéristiques numériques")
display(
    train[key_numeric].describe().T
    .rename(columns={
        "count": "Nombre", "mean": "Moyenne", "std": "Écart type", "min": "Minimum",
        "25%": "Q1", "50%": "Médiane", "75%": "Q3", "max": "Maximum",
    })
    .round(1)
)

# Boîtes à moustaches des principales caractéristiques (une échelle par variable).
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, column in zip(axes.flat, key_numeric):
    ax.boxplot(train[column].dropna(), **box_style)
    ax.set_title(column)
    ax.set_xticks([])
fig.suptitle("Boîtes à moustaches des principales caractéristiques numériques")
plt.tight_layout()
plt.show()

# Variables catégorielles : nombre de modalités et poids de la modalité dominante.
share_column = "Part de la modalité dominante (%)"
categorical_rows = []
for column in categorical_features:
    counts = train[column].value_counts(dropna=False)
    mode = counts.index[0]
    categorical_rows.append({
        "Variable": column,
        "Modalités observées": train[column].nunique(dropna=True),
        "Modalité dominante": "NA" if pd.isna(mode) else str(mode),
        share_column: round(counts.iloc[0] / len(train) * 100, 1),
    })
categorical_summary = (
    pd.DataFrame(categorical_rows)
    .sort_values(share_column, ascending=False)
    .reset_index(drop=True)
)
quasi_constant = categorical_summary[categorical_summary[share_column] >= 95]
show_title("Variables catégorielles quasi constantes")
print(f"{len(categorical_features)} variables catégorielles, dont "
      f"{len(quasi_constant)} dominées à 95 % ou plus par une seule modalité.")
display(quasi_constant)

# Modalités présentes dans test.csv mais jamais vues dans train.csv.
unseen_rows = []
for column in categorical_features:
    unseen = set(test[column].dropna()) - set(train[column].dropna())
    if unseen:
        unseen_rows.append({
            "Variable": column,
            "Modalités seulement dans test.csv": ", ".join(map(str, sorted(unseen))),
            "Maisons de test concernées": int(test[column].isin(unseen).sum()),
        })
show_title("Modalités présentes seulement dans test.csv")
display(pd.DataFrame(unseen_rows))

# Figure : modalité dominante (gauche) et nombre de modalités (droite).
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_share = categorical_summary.head(15)
axes[0].barh(
    [f"{name} = {mode}"
     for name, mode in zip(top_share["Variable"], top_share["Modalité dominante"])],
    top_share[share_column],
    color=np.where(top_share[share_column] >= 95, "tab:red", "tab:gray"),
)
axes[0].axvline(95, color="black", linestyle="--", linewidth=1, label="Seuil de 95 %")
axes[0].invert_yaxis()
axes[0].set_xlim(0, 100)
axes[0].set_title("Part de la modalité dominante (15 variables les plus concentrées)")
axes[0].set_xlabel("Maisons de train.csv (%)")
axes[0].legend(loc="lower left")

top_levels = categorical_summary.sort_values("Modalités observées", ascending=False).head(15)
axes[1].barh(top_levels["Variable"], top_levels["Modalités observées"], color="tab:orange")
axes[1].invert_yaxis()
axes[1].set_title("Nombre de modalités (15 variables les plus détaillées)")
axes[1].set_xlabel("Modalités observées dans train.csv")
plt.tight_layout()
plt.show()

Le prix médian est de 163 000 \$ (de 34 900 \$ à 755 000 \$), sous la moyenne de 180 921 \$ : quelques maisons chères étirent la distribution (asymétrie de 1,88). Les échelles des caractéristiques diffèrent fortement (`LotArea` jusqu'à 215 245 pi², `GarageCars` de 0 à 4), ce qui justifiera une standardisation pour les modèles linéaires.

Dans la figure, sept variables catégorielles (en rouge) sont dominées à plus de 95 % par une seule modalité et apportent peu d'information. À l'inverse, `Neighborhood` compte 25 modalités. Le code 150 de `MSSubClass` n'apparaît que dans `test.csv` : l'encodeur devra tolérer les catégories inconnues.

## 3. Identification des données manquantes

Le dictionnaire précise que, pour 14 variables, le code `NA` signifie l'**absence d'un équipement** : pas de ruelle, de sous-sol, de foyer, de garage, de piscine, de clôture ou d'équipement particulier. Pour les autres variables, une valeur absente correspond à une **information inconnue**. Nous distinguons ces deux situations, puis nous vérifions la cohérence entre les variables d'un même équipement : si `NA` signifie l'absence d'un garage, la surface du garage devrait par exemple être nulle.

In [ ]:
# Variables pour lesquelles data_description.txt définit NA comme une absence.
absence_columns = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature",
]

missing = pd.DataFrame({
    "Train (n)": train[features].isna().sum(),
    "Test (n)": test[features].isna().sum(),
})
missing = missing[missing.sum(axis=1) > 0].copy()
missing["Train (%)"] = (missing["Train (n)"] / len(train) * 100).round(1)
missing["Test (%)"] = (missing["Test (n)"] / len(test) * 100).round(1)
missing["Nature selon le dictionnaire"] = np.where(
    missing.index.isin(absence_columns),
    "NA = équipement absent", "Information inconnue",
)
missing = missing.sort_values(["Train (n)", "Test (n)"], ascending=False)
print(f"Variables avec au moins une valeur absente : {missing['Train (n)'].gt(0).sum()} "
      f"dans train, {missing['Test (n)'].gt(0).sum()} dans test.")
show_title("Valeurs absentes par variable")
display(missing)

# Cohérence entre les variables d'un même équipement.
basement_text = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
garage_text = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
consistency_rules = {
    "Garage absent (GarageType = NA) mais GarageArea > 0":
        lambda f: f["GarageType"].isna() & f["GarageArea"].gt(0),
    "GarageType renseigné, mais une autre information du garage absente":
        lambda f: f["GarageType"].notna() & f[garage_text + ["GarageYrBlt"]].isna().any(axis=1),
    "TotalBsmtSF > 0, mais une information du sous-sol absente":
        lambda f: f["TotalBsmtSF"].gt(0) & f[basement_text].isna().any(axis=1),
    "Fireplaces > 0 mais FireplaceQu = NA":
        lambda f: f["Fireplaces"].gt(0) & f["FireplaceQu"].isna(),
    "PoolArea > 0 mais PoolQC = NA":
        lambda f: f["PoolArea"].gt(0) & f["PoolQC"].isna(),
    "MasVnrType = None mais MasVnrArea > 0":
        lambda f: f["MasVnrType"].eq("None") & f["MasVnrArea"].gt(0),
    "GarageYrBlt postérieure à l'année de vente":
        lambda f: f["GarageYrBlt"].gt(f["YrSold"]),
    "YearRemodAdd antérieure à YearBuilt":
        lambda f: f["YearRemodAdd"].lt(f["YearBuilt"]),
}
show_title("Vérifications de cohérence entre variables d'un même équipement")
display(pd.DataFrame(
    [(rule, int(check(train).sum()), int(check(test).sum()))
     for rule, check in consistency_rules.items()],
    columns=["Vérification", "train.csv", "test.csv"],
))
show_title("Maisons de test.csv dont le garage est postérieur à la vente")
display(test.loc[test["GarageYrBlt"].gt(test["YrSold"]),
                 ["Id", "YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"]])

# Figure : proportion de valeurs absentes dans les deux fichiers.
plot_missing = (
    missing[missing["Train (n)"] > 0][["Train (%)", "Test (%)"]]
    .rename(columns={"Train (%)": "train.csv", "Test (%)": "test.csv"})
)
plot_missing.index = [
    f"{name} *" if name in absence_columns else name for name in plot_missing.index
]
plot_missing = (
    plot_missing.reset_index(names="Variable")
    .melt(id_vars="Variable", var_name="Fichier", value_name="Pourcentage")
)
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=plot_missing, y="Variable", x="Pourcentage", hue="Fichier", ax=ax)
ax.set_title("Proportion de valeurs absentes par variable")
ax.set_xlabel("Maisons concernées (%)")
ax.set_ylabel("")
ax.text(0.99, 0.02, "* NA signifie « équipement absent » selon le dictionnaire",
        transform=ax.transAxes, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

Dix-neuf variables ont des valeurs absentes dans `train.csv`, dans des proportions semblables à `test.csv`. Les plus touchées (marquées *) indiquent l'absence d'un équipement (piscine, ruelle, clôture…) : nous les remplacerons par une catégorie « Aucun ». Les vraies valeurs inconnues concernent surtout `LotFrontage` (17,7 %), à imputer à partir de l'entraînement seulement. Quelques incohérences de `test.csv` (équipements partiellement renseignés, garage construit en 2207) devront être corrigées.

## 4. Distribution des variables

Nous examinons la distribution de la cible, sur l'échelle des dollars et sur l'échelle logarithmique utilisée par la métrique Kaggle, puis l'asymétrie des 35 caractéristiques numériques. Nous repérons enfin les prix atypiques avec la règle de la boîte à moustaches : un prix est jugé atypique s'il est inférieur à Q1 − 1,5 × (Q3 − Q1) ou supérieur à Q3 + 1,5 × (Q3 − Q1), où Q1 et Q3 sont les 1er et 3e quartiles (méthode de Tukey).

In [ ]:
# Asymétrie des caractéristiques numériques.
skewness = pd.DataFrame({
    "Asymétrie": train[numeric_features].skew(),
    "Part de zéros (%)": (train[numeric_features].eq(0).mean() * 100),
}).round(2).sort_values("Asymétrie", ascending=False)
show_title("Caractéristiques numériques dont l'asymétrie dépasse 1")
print(f"{skewness['Asymétrie'].gt(1).sum()} variables numériques sur "
      f"{len(numeric_features)} ont une asymétrie supérieure à 1.")
display(skewness[skewness["Asymétrie"].gt(1)])

# Prix atypiques selon la règle de la boîte à moustaches.
q1, q3 = price.quantile([0.25, 0.75])
upper_limit = q3 + 1.5 * (q3 - q1)
log_q1, log_q3 = log_price.quantile([0.25, 0.75])
log_spread = log_q3 - log_q1
log_outside = log_price.lt(log_q1 - 1.5 * log_spread) | log_price.gt(log_q3 + 1.5 * log_spread)
print(f"Selon la méthode de Tukey, {price.gt(upper_limit).sum()} maisons dépassent le seuil haut de "
      f"SalePrice (Q3 + 1,5 × (Q3 − Q1) = {format_dollars(upper_limit)}), et {log_outside.sum()} "
      f"sortent des bornes calculées sur log(SalePrice).")

# Très grandes maisons.
show_title("Très grandes maisons de train.csv (GrLivArea > 4000)")
display(train.loc[
    train["GrLivArea"].gt(4000),
    ["Id", "GrLivArea", "SalePrice", "OverallQual", "Neighborhood", "SaleCondition"],
].sort_values("GrLivArea").style.hide(axis="index"))
show_title("Très grandes maisons de test.csv (GrLivArea > 4000)")
display(test.loc[
    test["GrLivArea"].gt(4000),
    ["Id", "GrLivArea", "OverallQual", "Neighborhood", "SaleCondition"],
].sort_values("GrLivArea").style.hide(axis="index"))

# Figure : cible (en dollars et en logarithme) et six caractéristiques numériques.
distribution_columns = [
    "GrLivArea", "LotArea", "TotalBsmtSF", "GarageArea", "LotFrontage", "YearBuilt",
]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flat

sns.histplot(price / 1000, bins=50, kde=True, ax=axes[0])
axes[0].axvline(price.mean() / 1000, color="tab:red", linestyle="--", label="Moyenne")
axes[0].axvline(price.median() / 1000, color="tab:green", linestyle=":", label="Médiane")
axes[0].set_title(f"SalePrice (asymétrie = {price.skew():.2f})")
axes[0].set_xlabel("Prix de vente (milliers de $)")
axes[0].legend()

sns.histplot(log_price, bins=50, kde=True, ax=axes[1], color="tab:purple")
axes[1].set_title(f"log(SalePrice) (asymétrie = {log_price.skew():.2f})")
axes[1].set_xlabel("Logarithme naturel du prix")

for ax, column in zip(axes[2:], distribution_columns):
    sns.histplot(train[column].dropna(), bins=40, ax=ax, color="tab:gray")
    ax.set_title(f"{column} (asymétrie = {train[column].skew():.2f})")
    ax.set_xlabel("pi²" if column != "YearBuilt" else "Année")
for ax in axes:
    ax.set_ylabel("Nombre de maisons")
fig.suptitle("Distribution de la cible et de six caractéristiques numériques (train.csv)", y=1.0)
plt.tight_layout()
plt.show()

Le prix est asymétrique vers la droite (1,88), alors que son logarithme est presque symétrique (0,12) : nous modéliserons `log(SalePrice)`, ce qui correspond à la métrique Kaggle. Parmi les caractéristiques, 18 sur 35 ont une asymétrie supérieure à 1 : surfaces étirées (`LotArea`, 12,21) ou variables surtout nulles (piscine, galeries). Une transformation `log1p` ou un indicateur de présence (exemple `hasPool`) pourra aider les modèles linéaires. Deux très grandes maisons (Id 524 et 1299) se vendent anormalement peu cher (section 6).

## 5. Analyse des corrélations

Nous mesurons l'association entre chaque caractéristique numérique et le prix avec deux coefficients, car un écart entre eux révèle une relation non linéaire ou tirée par quelques valeurs extrêmes :

- le **coefficient de Spearman**, calculé sur les rangs, mesure une relation monotone, même non linéaire, et est peu sensible aux valeurs extrêmes;
- le **coefficient de Pearson**, qui mesure une relation linéaire. Nous le calculons sur `SalePrice` et sur `log(SalePrice)`, l'échelle retenue pour la modélisation.

Les valeurs manquantes sont ignorées paire par paire. Nous recherchons ensuite les paires de caractéristiques fortement corrélées entre elles. Les variables catégorielles, absentes de ce calcul, sont étudiées graphiquement à la section 6.

In [ ]:
# Corrélation de chaque caractéristique numérique avec le prix.
correlations = pd.DataFrame({
    "Spearman (SalePrice)": train[numeric_features].corrwith(price, method="spearman"),
    "Pearson (log SalePrice)": train[numeric_features].corrwith(log_price),
    "Pearson (SalePrice)": train[numeric_features].corrwith(price),
}).round(3).sort_values("Spearman (SalePrice)", ascending=False)
show_title("Caractéristiques les plus corrélées au prix")
display(correlations.head(12))
show_title("Caractéristiques les moins corrélées au prix")
display(correlations.tail(4))

# Paires de caractéristiques numériques fortement corrélées entre elles.
feature_matrix = train[numeric_features].corr(method="spearman").abs()
upper = feature_matrix.where(np.triu(np.ones(feature_matrix.shape, dtype=bool), k=1))
show_title("Paires de caractéristiques fortement corrélées entre elles (|Spearman| ≥ 0,8)")
display(
    upper.stack()
    .loc[lambda values: values >= 0.8]
    .sort_values(ascending=False)
    .round(3)
    .rename("|Spearman|")
    .rename_axis(["Variable 1", "Variable 2"])
    .reset_index()
)

# Figure : matrice de Spearman des dix variables les plus liées au prix.
top_features = correlations.head(10).index.tolist()
spearman_matrix = train[top_features + ["SalePrice"]].corr(method="spearman")
mask = np.triu(np.ones_like(spearman_matrix, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(spearman_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Corrélation de Spearman"},
            ax=ax)
ax.set_title("Corrélations de Spearman entre les dix variables les plus liées au prix")
plt.tight_layout()
plt.show()

`OverallQual` (0,81) et `GrLivArea` (0,73) sont les caractéristiques les plus liées au prix, suivies du garage, de l'année de construction et du sous-sol. Quatre paires sont redondantes (Spearman ≥ 0,8) : `YearBuilt`–`GarageYrBlt` (0,89), `GarageCars`–`GarageArea` (0,85), `TotalBsmtSF`–`1stFlrSF` (0,83) et `GrLivArea`–`TotRmsAbvGrd` (0,83, hors de la carte). Pour un modèle linéaire, nous en garderons une par paire, les combinerons ou utiliserons une régularisation.

## 6. Visualisations pertinentes

Cette section réunit, dans une figure en quatre panneaux, les relations les plus utiles pour la suite du projet :

- **(a)** le prix selon la superficie habitable `GrLivArea`, colorée par la qualité, pour situer les maisons atypiques;
- **(b)** le prix selon la qualité globale `OverallQual`, la variable la plus corrélée au prix;
- **(c)** le prix par quartier `Neighborhood`, une variable catégorielle absente de l'analyse des corrélations;
- **(d)** le prix selon l'année de construction, en distinguant les ventes `Partial` (maisons neuves vendues avant la fin des travaux).

Les axes des prix des panneaux (a) à (c) sont en milliers de dollars; le panneau (d) utilise une échelle logarithmique.

In [ ]:
unusual = train["GrLivArea"].gt(4000) & price.lt(300_000)
partial = train["SaleCondition"].eq("Partial")
neighborhood_stats = (
    train.groupby("Neighborhood")["SalePrice"].agg(["count", "median"]).sort_values("median")
)
hood_order = neighborhood_stats.index.tolist()
decade = (train["YearBuilt"] // 10) * 10
# Conserver seulement les décennies d'au moins 10 ventes pour la courbe des médianes.
decade_counts = decade.value_counts()
reliable = decade.isin(decade_counts[decade_counts >= 10].index)
decade_median = price[reliable].groupby(decade[reliable]).median()

# (a) Superficie habitable.
fig, ax_area = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=train, x="GrLivArea", y=price / 1000, hue="OverallQual",
                palette="viridis", legend="full", alpha=0.75, s=25, ax=ax_area)
ax_area.scatter(train.loc[unusual, "GrLivArea"], price[unusual] / 1000, s=200,
                facecolors="none", edgecolors="tab:red", linewidths=2,
                label="GrLivArea > 4000 \net prix < 300 k$")
ax_area.set_title("(a) SalePrice selon le GrLivArea")
ax_area.set_xlabel("GrLivArea (pi²)")
ax_area.set_ylabel("SalePrice (milliers de $)")
ax_area.legend(
    title="OverallQual",
    ncol=1,
    fontsize=8,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)
plt.tight_layout()
plt.show()

# (b) Qualité globale.
fig, ax_qual = plt.subplots(figsize=(6, 4))
sns.boxplot(data=train, x="OverallQual", y=price / 1000, color="tab:blue",
            fliersize=3, ax=ax_qual)
for position, count in enumerate(train["OverallQual"].value_counts().sort_index()):
    ax_qual.text(position, 770, f"n={count}", ha="center", fontsize=9)
ax_qual.set_ylim(0, 800)
ax_qual.set_title("(b) SalePrice selon la OverallQual")
ax_qual.set_xlabel("OverallQual (1 = très faible, 10 = excellente)")
ax_qual.set_ylabel("SalePrice (milliers de $)")
plt.tight_layout()
plt.show()

# (c) Quartier.
fig, ax_hood = plt.subplots(figsize=(6, 6))
sns.boxplot(data=train, y="Neighborhood", x=price / 1000, order=hood_order,
            color="tab:orange", fliersize=2, ax=ax_hood)
ax_hood.set_yticks(range(len(hood_order)))
ax_hood.set_yticklabels([f"{name} (n={neighborhood_stats.loc[name, 'count']})"
                         for name in hood_order])
ax_hood.axvline(price.median() / 1000, color="black", linestyle="--", linewidth=1,
                label="Médiane globale")
ax_hood.set_title("(c) SalePrice par Neighborhood, triés selon la médiane")
ax_hood.set_xlabel("SalePrice (milliers de $)")
ax_hood.set_ylabel("")
ax_hood.legend(loc="upper right")
plt.tight_layout()
plt.show()

# (d) Année de construction. La médiane de chaque décennie est placée au centre
# de la décennie (1905 pour 1900-1909, etc.).
fig, ax_year = plt.subplots(figsize=(6, 5))
ax_year.scatter(train["YearBuilt"], price / 1000, alpha=0.3, s=15,
                color="tab:gray", label="Ventes")
ax_year.plot(decade_median.index + 5, decade_median / 1000, color="tab:red", marker="o",
             label="Médiane par décennie (>10 ventes)")
ax_year.set_yscale("log")
ax_year.set_yticks([50, 100, 200, 400, 700])
ax_year.set_yticklabels(["50", "100", "200", "400", "700"])
ax_year.yaxis.set_minor_formatter(NullFormatter())
ax_year.set_title("(d) SalePrice selon YearBuilt (échelle logarithmique)")
ax_year.set_xlabel("YearBuilt")
ax_year.set_ylabel("SalePrice (milliers de $)")
ax_year.legend(loc="upper left")
plt.tight_layout()
plt.show()

- **(a)** Le prix croît avec la surface. Les deux maisons encerclées (plus de 4 500 pi², qualité 10, ventes `Partial` dans `Edwards`) s'écartent nettement de la tendance. Nous comparerons l'entraînement avec et sans elles.
- **(b)** Le prix médian augmente à chaque niveau de qualité, de plus en plus vite (de 50 150 \$ à 432 390 \$) : une relation plutôt linéaire sur l'échelle logarithmique.
- **(c)** Les prix médians varient fortement entre les quartiers. Six quartiers comptent moins de 20 ventes et pourraient être regroupés.
- **(d)** Le prix reste stable pour les maisons construites avant 1980, puis augmente. L'âge à la vente sera une variable utile.

## 7. Synthèse et limites

### Principaux constats

| Constat | Élément de preuve | Piste pour le prétraitement (étape 4) |
|---|---|---|
| Prix asymétrique vers la droite | Asymétrie de 1,88, ramenée à 0,12 par le logarithme (section 4) | Modéliser `log(SalePrice)`, conforme à la métrique Kaggle |
| La plupart des `NA` indiquent une absence d'équipement | 14 variables; proportions semblables dans train et test (section 3) | Remplacer ces `NA` par une catégorie « Aucun » (ou 0 pour les surfaces) |
| Quelques vraies valeurs inconnues | `LotFrontage` (17,7 %), quelques valeurs dans test pour 15 variables complètes dans train (section 3) | Imputer dans un pipeline ajusté sur l'entraînement seulement |
| Incohérences ponctuelles | `GarageYrBlt` = 2207, équipements partiellement renseignés dans test (section 3) | Corriger les erreurs évidentes; vérifier la cohérence entre variables liées |
| Codes différents du dictionnaire | `BldgType`, `Exterior2nd`, `MSZoning`, `Neighborhood` (section 1) | Harmoniser `Exterior1st` et `Exterior2nd` avant toute combinaison |
| Variables numériques asymétriques ou surtout nulles | 18 variables sur 35 avec une asymétrie supérieure à 1 (section 4) | Transformation `log1p` ou indicateurs de présence, selon le modèle |
| Deux ventes atypiques | Id 524 et 1299 : plus de 4 500 pi², moins de 185 000 \$ (sections 4 et 6) | Comparer en validation croisée l'entraînement avec et sans ces maisons |
| Variables fortement associées au prix | `OverallQual` (0,81), `GrLivArea` (0,73), garage, année, sous-sol (section 5) | Variables candidates prioritaires pour le modèle |
| Caractéristiques redondantes | Quatre paires avec une corrélation de Spearman ≥ 0,8 (section 5) | Sélection, variables combinées (surface totale) ou régularisation |
| Catégories importantes, nombreuses ou rares | 25 quartiers; 7 variables quasi constantes; code 150 de `MSSubClass` seulement dans test (sections 2 et 6) | *One-hot* tolérant aux catégories inconnues; regroupement des modalités rares |
| Années difficiles à interpréter directement | Prix stable avant 1980, puis croissant (section 6) | Créer l'âge à la vente et le temps depuis la rénovation |

### Limites de l'analyse

- **Associations, pas causes.** Les relations observées ne démontrent pas qu'une caractéristique fait varier le prix. Par exemple, l'effet apparent de l'âge ou du quartier peut refléter la taille et la qualité des maisons.
- **Analyse principalement univariée et bivariée.** Nous avons examiné les variables une ou deux à la fois. Les interactions, par exemple entre la surface et le quartier, seront mieux évaluées par les modèles.
- **Petits effectifs.** Plusieurs catégories et niveaux de qualité comptent moins de 20 maisons. Les différences observées pour ces groupes sont incertaines.
- **Corrélations limitées aux variables numériques.** Les coefficients de Spearman et de Pearson ne mesurent pas l'association avec les catégories, que nous avons étudiées graphiquement pour quelques variables seulement.
- **Portée des données.** Les ventes ont eu lieu à Ames entre 2006 et 2010, en partie pendant la crise immobilière de 2008. Nos constats ne s'appliquent pas directement à un autre marché ou à une autre période.
- **Utilisation de `test.csv`.** Nous l'avons consulté pour comparer les valeurs manquantes et les catégories, mais aucune statistique de ce fichier ne servira à ajuster le prétraitement.

## Références

- [Consignes originales MAP6014](../docs/travail-de-mi-session-v2.pdf), étape 3. Le document porte le titre « travail de mi-session »; nous l'utilisons comme consigne du présent travail de fin de session.
- [Kaggle — House Prices – Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data), données de la compétition.
- [Description locale des variables](../data/data_description.txt), pour la signification des codes et des valeurs `NA`.
- [Étape 2 — Présentation du problème](02_presentation_probleme.ipynb), pour le contexte, la métrique et le dictionnaire des variables.

**Convention de dépôt.** Les sorties de calcul sont effacées avant le commit, conformément au [guide de collaboration](../docs/collaboration.md). Exécuter les cellules dans l'ordre avec le kernel `.venv` pour afficher les tableaux et les figures. Les valeurs citées dans les commentaires proviennent de l'exécution complète de ce notebook sur les fichiers Kaggle locaux.